In [ ]:
#| default_exp parallel

## Parallel notebook operations

Shared locks for MCP and notebook helpers. Calls touching the same notebook are serialized, calls touching different notebooks can proceed independently, and notebook execution uses one global semaphore.

Parallelism is useful only if notebook writes remain deterministic. This module gives MCP tools per-notebook locks for read/write operations and a single execution slot for running notebooks, because kernels and output writes are much harder to reason about concurrently.

```python
with notebook_locks("nbs/01_read.ipynb", "nbs/02_write.ipynb"):
    ...

with execution_slot():
    ...
```

In [ ]:
from contextlib import redirect_stdout as _redirect_stdout
from io import StringIO as _StringIO
import threading
import time
from nbskill.parallel import execution_slot as _example_execution_slot
from nbskill.parallel import notebook_key as _example_notebook_key
from nbskill.parallel import notebook_locks as _example_notebook_locks
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook

In [ ]:
print(_example_notebook_key("nbs/../nbs/01_read.ipynb"))
with _example_notebook_locks("nbs/01_read.ipynb", "nbs/02_write.ipynb"):
    print("notebook locks acquired")
with _example_execution_slot():
    print("execution slot acquired")

NbskillTimeoutSkipped: nbskill: skipped cell id=ex_4d4bc5; it previously exceeded the 30s timeout. Edit the cell to change its hash and rerun it.

In [ ]:
#| export
import threading
from contextlib import contextmanager
from pathlib import Path

### Shared coordination state

The lock registry and execution semaphore live in one small module so every notebook operation uses the same coordination policy.

In [ ]:
#| export
_LOCKS_GUARD = threading.Lock()
_NOTEBOOK_LOCKS = {}
_EXECUTION_SEMAPHORE = threading.Semaphore(1)

### Stable path keys

Notebook paths can be relative, absolute, or not created yet. `notebook_key` normalizes them into a consistent lock key without requiring the file to already exist.

In [ ]:
#| export
def notebook_key(path):
    "Return a stable lock key for a notebook path."
    if path is None: return None
    return str(Path(path).expanduser().resolve(strict=False))

In [ ]:
#| export
def _notebook_lock(key):
    with _LOCKS_GUARD:
        lock = _NOTEBOOK_LOCKS.get(key)
        if lock is None:
            lock = threading.RLock()
            _NOTEBOOK_LOCKS[key] = lock
        return lock

### Per-notebook locks

`notebook_locks` acquires all requested notebook locks in sorted order. That lets operations touching multiple notebooks avoid deadlocks while still allowing unrelated notebooks to be edited in parallel.

In [ ]:
#| export
@contextmanager
def notebook_locks(*paths):
    "Acquire per-notebook locks in a stable order."
    keys = sorted({notebook_key(path) for path in paths if notebook_key(path) is not None})
    locks = [_notebook_lock(key) for key in keys]
    for lock in locks: lock.acquire()
    try:
        yield
    finally:
        for lock in reversed(locks): lock.release()

### One execution slot

Notebook execution mutates kernels and outputs, so execution is serialized globally. Read and write operations can still use per-notebook locks around their own files.

In [ ]:
#| export
@contextmanager
def execution_slot():
    "Serialize notebook execution across parallel MCP calls."
    _EXECUTION_SEMAPHORE.acquire()
    try:
        yield
    finally:
        _EXECUTION_SEMAPHORE.release()

In [ ]:
root = demo_path("09_parallel_keys")
try:
    root.mkdir()
    a = root / "a.ipynb"
    b = root / "sub" / ".." / "b.ipynb"
    assert notebook_key(a).endswith("a.ipynb")
    assert notebook_key(b).endswith("b.ipynb")
    assert notebook_key(None) is None
finally:
    remove_demo_path(root)

In [ ]:
path = demo_path("09_parallel_same.ipynb")
try:
    entered = []
    def enter_same_lock():
        with notebook_locks(path):
            entered.append(True)
    with notebook_locks(path):
        thread = threading.Thread(target=enter_same_lock)
        thread.start()
        time.sleep(0.05)
        assert entered == []
    thread.join(timeout=1)
    assert entered == [True]
finally:
    remove_demo_path(path)

In [ ]:
import threading
import time

active = 0
max_active = 0
guard = threading.Lock()

def hold_execution_slot():
    global active, max_active
    with execution_slot():
        with guard:
            active += 1
            max_active = max(max_active, active)
        time.sleep(0.05)
        with guard:
            active -= 1

threads = [threading.Thread(target=hold_execution_slot) for _ in range(3)]
for thread in threads: thread.start()
for thread in threads: thread.join()
assert max_active == 1

In [ ]:
assert _example_notebook_key("nbs/../nbs/01_read.ipynb") == _example_notebook_key("nbs/01_read.ipynb")
_acquired = False
with _example_notebook_locks("nbs/01_read.ipynb", "nbs/01_read.ipynb", None):
    _acquired = True
assert _acquired